In [0]:
df = spark.table("`databrucks-ci-cd-pipelines`.sathish_skyuv.order_details")
display(df)



In [0]:
from pyspark.sql import functions as F

df_refined = (
    df
    # Cast amount: remove commas then convert to double
    .withColumn("amount", F.regexp_replace(F.col("amount"), ",", "").cast("double"))
    # Standardise status: lowercase + fix known abbreviations
    .withColumn(
        "status",
        F.when(F.lower(F.col("status")).isin("pend"), F.lit("pending"))
         .otherwise(F.lower(F.col("status")))
    )
    # Parse date / timestamp strings
    .withColumn("order_date", F.try_to_timestamp(F.col("order_date"), F.lit("yyyy-MM-dd")).cast("date"))
    .withColumn("updated_at", F.try_to_timestamp(F.col("updated_at"), F.lit("yyyy-MM-dd'T'HH:mm:ss'Z'")))
    # Compute net amount after discount and shipping
    .withColumn("net_amount", F.round(F.col("amount") - F.col("discount_amount") + F.col("shipping_amount"), 2))
)

display(df_refined)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS `databrucks-ci-cd-pipelines`.silver")

df_refined.write.mode("overwrite").saveAsTable("`databrucks-ci-cd-pipelines`.silver.order_details_silver")

print("Wrote", df_refined.count(), "rows to `databrucks-ci-cd-pipelines`.silver.order_details_silver")